# Lightcone kSZ — Exploration

Loads products from `data/products/`. Run `scripts/01_make_ksz_lightcone_maps.py` first.
Heavy functions live in `src/ksz_pipeline/`. This notebook is for plotting only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from ksz_pipeline.plotting.styles import PNG_STYLE, PDF_STYLE, save_pdf_png

## Load products

In [ ]:
ksz_map = np.load('data/products/ksz_map_lightcone.npy')
Dl_data = np.load('data/products/ksz_Dl_lightcone.npz')
reion   = np.load('data/products/lightcone_reion_history.npz')
ell, Dl, Dl_err = Dl_data['ell'], Dl_data['Dl'], Dl_data['Dl_err']
z, xe, tau      = reion['z'], reion['xe'], reion['tau']
print(f'kSZ map shape: {ksz_map.shape}  RMS: {np.sqrt(np.mean(ksz_map**2)):.4e}')

## kSZ map

In [ ]:
BOX_LEN = 800.0  # Mpc
vmax = np.percentile(np.abs(ksz_map), 99)
with mpl.rc_context(PNG_STYLE):
    fig, ax = plt.subplots(figsize=(8,8), constrained_layout=True)
    im = ax.imshow(ksz_map.T, cmap='seismic_r', origin='lower',
                   extent=[0,BOX_LEN,0,BOX_LEN], vmin=-vmax, vmax=vmax, aspect='equal')
    fig.colorbar(im, ax=ax, label=r'$\Delta T/T|_{\rm kSZ}$')
    ax.set_xlabel('x [Mpc]'); ax.set_ylabel('y [Mpc]')
    plt.show()

## D_ell

In [ ]:
with mpl.rc_context(PNG_STYLE):
    fig, ax = plt.subplots(figsize=(10,7), constrained_layout=True)
    ax.errorbar(ell, Dl, yerr=Dl_err, fmt='s-', color='darkblue',
                lw=1.5, ms=4, capsize=3, label=r'This work: Lightcone $D_\ell$')
    ax.errorbar(3000, 1.1, yerr=[[0.7],[1.0]], fmt='s', ms=8,
                capsize=5, color='red', label='Reichardt+2021')
    for fname, color, ls, label in [
        ('Georgiev_kSZ_zend_slow.csv',  'gray',    '--', 'Georgiev+24 (Slow)'),
        ('Georgiev_kSZ_zend_mid.csv',   'black',   '-',  'Georgiev+24 (Mid)'),
        ('Georgiev_kSZ_zend_rapid.csv', 'darkred', ':',  'Georgiev+24 (Rapid)'),
    ]:
        try:
            df = pd.read_csv(fname)
            ax.plot(df.iloc[:,0], df.iloc[:,1], color=color, ls=ls, label=label, alpha=0.6)
        except FileNotFoundError:
            pass
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'Multipole $\ell$'); ax.set_ylabel(r'$D_\ell\ [\mu{\rm K}^2]$')
    ax.set_xlim(1e2,1e4); ax.set_ylim(1e-2,1e2)
    ax.legend(loc='upper left'); plt.show()

## Reionization history

In [ ]:
with mpl.rc_context(PNG_STYLE):
    fig, axes = plt.subplots(1,2, figsize=(14,6), constrained_layout=True)
    axes[0].plot(z, xe, lw=2)
    axes[0].set_xlabel('Redshift z'); axes[0].set_ylabel(r'$x_e$')
    axes[0].invert_xaxis()
    axes[1].plot(z, tau, lw=2, color='darkorange')
    axes[1].axhline(0.054, color='gray', ls='--', label='Planck 2018')
    axes[1].set_xlabel('Redshift z'); axes[1].set_ylabel(r'$\tau(<z)$')
    axes[1].invert_xaxis(); axes[1].legend()
    plt.show()